# 02 — Parameter Sweep

Sensitivity analysis of key parameters using SimulationRunner.run_sweep().

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

from simulation.runner import SimulationRunner, RunConfig
from model.config import EcosystemConfig

runner = SimulationRunner()
base_cfg = EcosystemConfig(width=30, height=30, initial_sheep=80, initial_wolves=30)
print('Setup complete.')

## 1. Single Parameter Sweep — sheep_reproduce

We sweep sheep reproduction rate from 0.01 to 0.15 across 8 values, with 3 replicate seeds each.

In [ ]:
import numpy as np

sweep_values = list(np.linspace(0.01, 0.15, 8).round(3))
seeds = [42, 43, 44]

sweep_results = runner.run_sweep(
    base_config=base_cfg,
    param_name='sheep_reproduce',
    param_values=sweep_values,
    n_steps=300,
    seeds=seeds,
)
print(f'Total runs: {len(sweep_results)}')

In [ ]:
# Aggregate: mean peak wolf count per sheep_reproduce value
records = []
for r in sweep_results:
    records.append({
        'sheep_reproduce': r.config.ecosystem_config.sheep_reproduce,
        'peak_wolves': r.data['Wolves'].max(),
        'extinct': len(r.extinction_events) > 0,
    })

agg = pd.DataFrame(records).groupby('sheep_reproduce').agg(
    mean_peak_wolves=('peak_wolves', 'mean'),
    std_peak_wolves=('peak_wolves', 'std'),
    extinct_rate=('extinct', 'mean'),
).reset_index()

fig = go.Figure([
    go.Scatter(
        x=agg['sheep_reproduce'], y=agg['mean_peak_wolves'],
        error_y=dict(type='data', array=agg['std_peak_wolves'].fillna(0)),
        mode='lines+markers', name='Mean Peak Wolves',
        line=dict(color='red'),
    )
])
fig.update_layout(
    title='Mean Peak Wolf Population vs Sheep Reproduce Rate (3 replicates)',
    xaxis_title='sheep_reproduce', yaxis_title='Mean Peak Wolf Count',
)
fig.show()
print(agg.to_string(index=False))

### Interpreting the threshold

There is typically a critical **sheep_reproduce** threshold below which wolves cannot maintain a viable population — sheep reproduce too slowly to sustain predation, causing wolf starvation.

Above the threshold, wolf peak count rises steeply before saturating, because grass regrowth becomes the binding constraint: sheep may reproduce quickly but cannot eat bare ground.

The error bands show variability across seeds — wide bands at the threshold indicate the system is near a **tipping point**: small perturbations determine whether wolves survive or go extinct. This is ecologically meaningful; real systems near tipping points are disproportionately sensitive to stochastic events (disease, weather, habitat fragmentation).

## 2. Two-Parameter Grid Sweep — stability region

We cross sheep_reproduce × wolf_reproduce across 5×5 = 25 combinations and plot extinction rate.

In [ ]:
sheep_vals = list(np.linspace(0.02, 0.12, 5).round(3))
wolf_vals  = list(np.linspace(0.01, 0.10, 5).round(3))
grid_seeds = [42, 43, 44]

grid_records = []
for sv in sheep_vals:
    for wv in wolf_vals:
        cfg = EcosystemConfig(
            width=20, height=20, initial_sheep=60, initial_wolves=20,
            sheep_reproduce=sv, wolf_reproduce=wv,
        )
        runs = runner.run_sweep(
            base_config=cfg,
            param_name='seed',        # dummy — we just want replicates
            param_values=[42],        # one 'value', but multiple seeds
            n_steps=300,
            seeds=grid_seeds,
        )
        extinct_rate = sum(1 for r in runs if r.extinction_events) / len(runs)
        grid_records.append({'sheep_reproduce': sv, 'wolf_reproduce': wv, 'extinct_rate': extinct_rate})

grid_df = pd.DataFrame(grid_records)
pivot = grid_df.pivot(index='wolf_reproduce', columns='sheep_reproduce', values='extinct_rate')

fig2 = go.Figure(go.Heatmap(
    z=pivot.values,
    x=[str(v) for v in pivot.columns],
    y=[str(v) for v in pivot.index],
    colorscale='RdYlGn_r',
    colorbar=dict(title='Extinction Rate'),
    zmin=0, zmax=1,
))
fig2.update_layout(
    title='Extinction Rate — sheep_reproduce × wolf_reproduce (3 seeds, 300 steps)',
    xaxis_title='sheep_reproduce', yaxis_title='wolf_reproduce',
)
fig2.show()

### Identifying the stability region

The heatmap reveals a **stability region** — parameter combinations where neither wolves nor sheep go extinct within 300 steps. Expect to find it in the middle of the grid:

- **High wolf_reproduce + low sheep_reproduce**: wolves over-reproduce and exhaust sheep → sheep extinction → wolf starvation.
- **Low wolf_reproduce + high sheep_reproduce**: wolves cannot sustain numbers; sheep eventually graze grass to bare ground without predation control.
- **Stable zone**: balanced reproduction rates where predator-prey oscillation persists.

In real conservation planning, this kind of sensitivity map informs **stocking density decisions** — for example, how many Sparrowhawks to reintroduce to a habitat relative to the estimated small-bird prey population.